# Nemotron Relational on AdventureWorks, from Unity Catalog

Four questions of the AdventureWorks sample schema -- demand
forecasting, customer inactivity, product recommendation, and filling in
a missing attribute -- answered by six predictions, since the forecast is
asked three ways. No feature engineering and no training step.

Nothing is copied out of the warehouse. Sampling is compiled to SQL and
runs as you, so row and column permissions, lineage and audit all apply,
and only the sampled rows are sent to the endpoint.

Needs a Nemotron Relational serving endpoint, a SQL warehouse, and DBR 15.4 LTS.

The cell below is the whole install. `--find-links` points pip at the
Unity Catalog volume the `nemotron-predict-client`, `nemotron_relational` and `nemotron-predict-connectors`
wheels were uploaded to; adjust that path to match your workspace. None
of those three names exist on PyPI, so nothing else can satisfy them.

The extras pull exactly what this notebook imports -- `databricks` for
the SQL connector behind `Graph.from_databricks`, `databricks-serving`
for the workspace SDK behind `PredictClient.for_databricks_serving`, and
`nemotron_relational` for the engine itself. Every version bound they need is
already declared in the SDK's own metadata, so there is nothing to
restate here.

`numpy<2` is the one bound that is not the SDK's, and it is specific to
this runtime: DBR 15.4 LTS ships numpy 1.23.5, which is below the SDK's
`numpy>=1.24` floor, so pip does upgrade it -- and left unbounded it
would reach 2.x, whose C ABI the runtime's compiled modules are not
built against.

**Runs inside a Databricks workspace.** It uses `dbutils` and the
workspace client, which exist only in that runtime, so it cannot be
executed from a local interpreter.

**Before you start**, load the AdventureWorks tables into the catalog and
schema named in the widgets below, and set the endpoint widget to a served
Nemotron Relational model.


In [ ]:
%pip install --find-links /Volumes/main/default/wheels 'nemotron-predict-client[nemotron_relational,databricks,databricks-serving]' 'numpy<2'

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text('catalog', 'main', 'Catalog')
dbutils.widgets.text('schema', 'kumo_rfm', 'Schema')
dbutils.widgets.text('endpoint', '', 'Nemotron Relational endpoint')
dbutils.widgets.text('warehouse_id', '', 'SQL warehouse ID')
dbutils.widgets.text('output_table', 'aw_predictions', 'Output table')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
endpoint = dbutils.widgets.get('endpoint')
warehouse_id = dbutils.widgets.get('warehouse_id')
output_table = dbutils.widgets.get('output_table')

for value, name in ((endpoint, 'endpoint'), (warehouse_id, 'warehouse_id')):
    if not value:
        raise ValueError(f'Set the {name} widget')

## 1. Connect

No token anywhere. `credentials_provider` hands the connector the
notebook's own identity, which the connector prefers over an
`access_token` and re-invokes as credentials refresh. The endpoint is
addressed the same way, so nothing here needs a secret.

The warehouse is named by ID; its hostname and HTTP path are read back
from the workspace rather than pasted out of the UI.

In [ ]:
from databricks import sql as dbsql
from databricks.sdk import WorkspaceClient
from nemotron_predict import PredictClient
from nemotron_predict.relational import Graph

workspace = WorkspaceClient()
odbc = workspace.warehouses.get(warehouse_id).odbc_params

# Opened here rather than inside from_databricks, so the same connection --
# one identity, one compute path -- also serves the queries further down.
connection = dbsql.connect(
    server_hostname=odbc.hostname,
    http_path=odbc.path,
    credentials_provider=lambda: workspace.config.authenticate,
)


def query(sql):
    with connection.cursor() as cursor:
        cursor.execute(sql)
        return cursor.fetchall()

## 2. Build the graph

The tables are stored as `aw_*`; `source_name` exposes each under the
short name the queries use, because PQL reads the catalog's spelling and
is case-sensitive.

No keys and no edges are listed. Unity Catalog already declares them as
`PRIMARY KEY` and `FOREIGN KEY` constraints, and those are read directly.
Declaring them in the catalog is worth doing for its own sake -- every
tool sees them, not just this notebook.

`sales_order_details` carries `CustomerID` and `OrderDate` denormalised
from its parent order. The recommendation query in step 5 needs a direct
foreign key to `customers`; routing through `sales_order_headers` is
rejected. A view over the raw detail table would do just as well.

In [ ]:
graph = Graph.from_databricks(
    connection=connection,
    catalog=catalog,
    schema=schema,
    tables=[
        {'name': 'customers', 'source_name': 'aw_customers'},
        {'name': 'products', 'source_name': 'aw_products'},
        {
            'name': 'sales_order_headers',
            'source_name': 'aw_sales_order_headers',
        },
        {
            'name': 'sales_order_details',
            'source_name': 'aw_sales_order_details',
        },
    ],
)

## 3. Say which tables are timelines

This part the catalog cannot tell us: a `TIMESTAMP` column says when a
row was written, not that the table records events. It has to be stated,
and stating it is not optional here -- `OrderDate` and `ModifiedDate` on
the detail table span exactly the same range, so the inference picks
between them arbitrarily and picks wrong.

Setting it to `None` marks a table as a lookup, so its dates are treated
as attributes rather than as history.

In [ ]:
graph['sales_order_headers'].time_column = 'OrderDate'
graph['sales_order_details'].time_column = 'OrderDate'
graph['customers'].time_column = None
graph['products'].time_column = None

# OrderQty is left as inferred, which is `categorical` -- a column counts as
# categorical when its distinct values are few relative to its rows, and an
# order line is almost always for one or two items. Declaring it `numerical`
# looks obviously right and makes the forecast measurably worse: on this data
# it costs about a third of the predicted value, so the query parser's
# semantic-type warning is worth reading but not worth acting on here.

# rowguid is a per-row UUID that AdventureWorks carries for replication. It
# identifies a row and says nothing about it, and every column of the source
# table is read, so it arrives as a feature unless dropped.
#
# This is not tidiness. Left on sales_order_details -- the table the demand
# query aggregates over -- it drives the regression prediction to exactly 0.00
# at every anchor and seed. Removed, the same query on the same data returns
# 200-285 against an actual of 356. On the other three tables it is harmless,
# but a column that is unique per row cannot inform a prediction anywhere, so
# all four go.
for table_name in (
    'customers',
    'products',
    'sales_order_headers',
    'sales_order_details',
):
    graph[table_name].remove_column('rowguid')

graph.print_metadata()
graph.print_links()
graph.validate()

## 4. Forecast 30-day demand

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)  ← how many will sell
FOR EACH products.ProductID                             ← one row per product
```

The entities are picked by query rather than written in, so this survives
the sample data being republished with different ids.

Expect a low number here, and do not read it as the model failing. With
no anchor given, the prediction is made from the last timestamp in the
data, and this extract stops mid-decline: 92% of products sold nothing
in its final 30 days. The historical anchor below is the more useful
demonstration, because it sits in a live part of the history and its
answer can be checked against what actually happened.

In [ ]:
import pandas as pd

client = PredictClient.for_databricks_serving(
    endpoint, workspace_client=workspace
)
model = client.relational(graph)

top_product = query(f"""
    SELECT ProductID FROM `{catalog}`.`{schema}`.aw_sales_order_details
    GROUP BY ProductID ORDER BY sum(OrderQty) DESC LIMIT 1
""")[0][0]

demand = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FOR EACH products.ProductID'
)

# The model reads a row's features in order, so the answer moves when the
# columns do -- which is why column_shuffle exists: it shuffles them per
# estimator so the ensemble averages the effect out. It is off by default and
# num_estimators is 1, which is the most sensitive setting there is. Measured
# here over five column permutations, everything else fixed:
#
#     default (1 estimator, no shuffle)   115.5 - 266.6   2.31x
#     4 estimators, shuffled              221.7 - 251.9   1.14x
#
# Four is the maximum. The cost is four forward passes per request.
ENSEMBLE = {'num_estimators': 4, 'column_shuffle': True}

results = {
    'demand_30d': model.predict(
        demand,
        indices=[top_product],
        run_mode='fast',
        inference_config=ENSEMBLE,
    )
}
display(results['demand_30d'])

### The same forecast, as of a past date

Predictions default to the latest timestamp in the graph. An anchor
inside the history asks what the answer would have been then, using only
what was known before it. Derived from the data, because this sample has
been republished with shifted dates more than once and a fixed date
quietly falls outside it.

In [ ]:
latest = query(
    f'SELECT max(OrderDate) FROM `{catalog}`.`{schema}`.aw_sales_order_details'
)[0][0]
anchor = pd.Timestamp(latest) - pd.Timedelta(days=180)
print(f'latest order {latest}; anchoring at {anchor.date()}')

results['demand_30d_historical'] = model.predict(
    demand,
    indices=[top_product],
    run_mode='fast',
    anchor_time=anchor,
    inference_config=ENSEMBLE,
)
display(results['demand_30d_historical'])

### Several timeframes at once

In [ ]:
results['demand_3_timeframes'] = model.predict(
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FORECAST 3 TIMEFRAMES FOR EACH products.ProductID',
    indices=[top_product],
    run_mode='fast',
    inference_config=ENSEMBLE,
)
display(results['demand_3_timeframes'])

## 5. Who is about to go quiet, and what to offer them

In [ ]:
recent_customers = [
    row[0]
    for row in query(f"""
    SELECT CustomerID FROM `{catalog}`.`{schema}`.aw_sales_order_headers
    GROUP BY CustomerID ORDER BY count(*) DESC LIMIT 25
""")
]

results['inactivity_90d'] = model.predict(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days)=0 '
    'FOR EACH customers.CustomerID',
    indices=recent_customers,
    run_mode='fast',
)
display(results['inactivity_90d'])

Recommendation ranks products a customer has not bought yet. This is a
link-prediction query: at most 200 entities per request, against 1,000
for the others, and `RANK TOP k` is capped at 20.

In [ ]:
results['recommend_top10'] = model.predict(
    'PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days) '
    'RANK TOP 10 FOR EACH customers.CustomerID',
    indices=recent_customers[:10],
    run_mode='fast',
)
display(results['recommend_top10'])

## 6. Fill in a missing attribute

No time range, so this asks about the row as it stands rather than about
the future.

`Color` is inferred `categorical` here, which is what makes it a legal
target -- a `text` column cannot be predicted and the query is refused
before it is sent. Inference reads a 1,000-row sample, so on a larger
products table state it: `graph['products']['Color'].stype =
nemotron_relational.Stype.categorical`.

In [ ]:
results['product_color'] = model.predict(
    'PREDICT products.Color FOR EACH products.ProductID',
    indices=[top_product],
    run_mode='fast',
)
display(results['product_color'])

## 7. Save the predictions

Run as a job, `display()` output is discarded. Anything worth reading
afterwards has to be written somewhere.

In [ ]:
from pyspark.sql.functions import current_timestamp

# One row per predicted entity, with the row itself as JSON. The six queries
# return six different shapes -- a regression has no class probabilities, a
# ranking has no PREDICTED, a forecast adds FORECAST_STEP -- and appending
# them into one table by union drops whatever the first write did not define.
# Keeping the payload opaque keeps every column, whatever the task returns.
rows = [
    (name, position, frame.iloc[position].to_json())
    for name, frame in results.items()
    for position in range(len(frame))
]

target = f'`{catalog}`.`{schema}`.{output_table}'
(
    spark.createDataFrame(
        rows, 'prediction string, row_index int, payload string'
    )
    .withColumn('scored_at', current_timestamp())
    .write.mode('append')
    .saveAsTable(target)
)
print(f'wrote {len(rows)} rows across {len(results)} predictions to {target}')

In [ ]:
client.close()
connection.close()